# 07 — SAM 3: Prompts de texto

## ¿Qué vamos a construir hoy?

Segmentarás objetos describiendo en texto lo que buscas — sin necesitar un detector
previo como YOLO. SAM 3 entiende el lenguaje natural y encuentra todas las instancias
del concepto que le pidas en la imagen.

**Aprenderás a:**
- Usar `SAM3SemanticPredictor` para prompts de texto
- Comparar texto vs. bounding box: cuándo conviene cada uno
- Ajustar el umbral de confianza para controlar cuántas detecciones obtienes
- Ver el efecto de conceptos específicos vs. genéricos

**Tiempo estimado:** 30 minutos

## ⏱️ Estructura de la Clase (Duración estimada: 1 hora)
- **Introducción y Conceptos Base**: 15 min
- **Desarrollo y Demostración Práctica**: 25 min
- **Análisis y Casos Extremos (Pausa y Observa)**: 20 min

## Texto como prompt

En NB06 le dábamos a SAM las coordenadas de una caja: *"el objeto está aquí, segméntalo"*.
Con SAM 3, también puedes decirle: *"encuentra y segmenta todo lo que sea una persona"*.

```
Prompt bbox:  SAM(image, bboxes=[[100,50,300,400]])  ← tú ya sabes dónde está
Prompt texto: predictor(text=["person"])             ← SAM lo busca solo
```

Cuándo usar cada uno:
- **Texto** → no tienes detector previo, describes lo que buscas en lenguaje natural.
- **Bbox** → ya tienes un detector entrenado para tu dominio y quieres mayor precisión.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
sam_path = "/content/drive/MyDrive/RandD/Archive_Zero_Resolved/sam3.pt"

In [ ]:
!pip install supervision ultralytics
import supervision as sv
from ultralytics.models.sam import SAM3SemanticPredictor
from ultralytics import YOLO, SAM
import torch
import cv2
import matplotlib.pyplot as plt
import urllib.request
from pathlib import Path

Path("assets").mkdir(exist_ok=True)
urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg",    "assets/bus.jpg")
urllib.request.urlretrieve("https://ultralytics.com/images/zidane.jpg", "assets/zidane.jpg")

image = cv2.imread("assets/bus.jpg")
print(f"Imagen cargada: {image.shape}")


## Paso 1: Cargar SAM3SemanticPredictor

`SAM3SemanticPredictor` es el predictor especializado en prompts de texto y conceptos.
Es distinto al `SAM()` genérico que usamos en NB06 para bboxes.


## Los parámetros de `overrides`

El diccionario `overrides` configura cómo funciona el predictor:

| Parámetro | Valor | Significado |
|-----------|-------|-------------|
| `conf` | `0.25` | Umbral de confianza mínimo. Solo se incluyen detecciones con score ≥ 0.25. Sube este valor para reducir falsos positivos; bájalo para capturar más objetos. |
| `task` | `"segment"` | Le dice a Ultralytics que el objetivo es segmentación (no detección simple ni clasificación). |
| `mode` | `"predict"` | Modo inferencia. Otras opciones son `"train"` y `"val"`, pero para usar el modelo siempre es `"predict"`. |
| `model` | `"sam3.pt"` | Archivo de pesos del modelo. Si lo tienes en otra ruta, cambia este valor. |

El parámetro opcional `half=True` activa precisión FP16 (solo en GPU), lo que
reduce el consumo de memoria y acelera la inferencia sin pérdida significativa de precisión.

In [ ]:
overrides = dict(conf=0.25, task="segment", mode="predict", model=sam_path)
if torch.cuda.is_available():
    overrides["half"] = True   # FP16 solo en GPU

predictor = SAM3SemanticPredictor(overrides=overrides)
print("Predictor listo")


## Paso 2: Inferencia con texto

`predictor.set_image()` carga la imagen en memoria una sola vez.
Después puedes hacer varias llamadas con distintos prompts sin recargarla.


In [ ]:
predictor.set_image(image)

resultados = predictor(text=["person"])[0]
detections = sv.Detections.from_ultralytics(resultados)

print(f"Objetos encontrados: {len(detections)}")
print(f"¿Tiene máscaras?    {detections.mask is not None}")


## Paso 3: Visualizar


In [ ]:
box_annotator  = sv.BoxAnnotator()
mask_annotator = sv.MaskAnnotator(opacity=0.6)

annotated = mask_annotator.annotate(scene=image.copy(), detections=detections)
annotated = box_annotator.annotate(scene=annotated,    detections=detections)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title('SAM 3 con prompt de texto: "person"')
plt.show()


## Pausa y observa: confianza por objeto

A diferencia del pipeline YOLO → SAM de NB06, aquí SAM 3 asigna
directamente un score de confianza a cada máscara.


In [ ]:
if detections.confidence is not None:
    for i, conf in enumerate(detections.confidence):
        area = int(detections.mask[i].sum()) if detections.mask is not None else 0
        print(f"Objeto {i}: confianza={conf:.3f}  area={area:,} px2")


## 🔧 Exploración interactiva

### Experimento 1: Concepto específico vs. genérico

¿SAM 3 devuelve más o menos objetos cuando el concepto es más amplio?


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, concepto in zip(axes, ["vehicle", "bus", "person"]):
    res = predictor(text=[concepto])[0]
    det = sv.Detections.from_ultralytics(res)
    scn = sv.MaskAnnotator(opacity=0.6).annotate(scene=image.copy(), detections=det)
    ax.imshow(cv2.cvtColor(scn, cv2.COLOR_BGR2RGB))
    ax.set_title(f'"{concepto}" — {len(det)} objetos')
    ax.axis("off")
plt.tight_layout()
plt.show()
# 💭 Reflexión: ¿"vehicle" incluye el bus? ¿Qué ocurre con conceptos ambiguos?
# Conceptos amplios capturan más instancias pero pueden incluir falsos positivos.


### Experimento 2: Umbral de confianza

`conf` controla el umbral mínimo para incluir una detección.
Podemos variarlo sin recargar el modelo.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, umbral in zip(axes, [0.1, 0.3, 0.6]):
    predictor.args.conf = umbral
    res = predictor(text=["person"])[0]
    det = sv.Detections.from_ultralytics(res)
    scn = sv.MaskAnnotator(opacity=0.6).annotate(scene=image.copy(), detections=det)
    ax.imshow(cv2.cvtColor(scn, cv2.COLOR_BGR2RGB))
    ax.set_title(f"conf={umbral}  ({len(det)} objetos)")
    ax.axis("off")
plt.tight_layout()
plt.show()
predictor.args.conf = 0.25   # restaurar valor original
# 💭 Reflexión: ¿Qué umbral produce el mejor resultado visual?
# Un umbral bajo puede incluir detecciones parciales en los bordes de la imagen.


### Experimento 3: Texto vs. bounding box

Compara las máscaras de SAM 3 cuando la guía es texto
versus cuando son cajas de YOLO (como en NB06).


In [ ]:
predictor.args.conf = 0.25
res_texto   = predictor(text=["person"])[0]
det_texto   = sv.Detections.from_ultralytics(res_texto)

yolo_model  = YOLO("yolov8n.pt")
yolo_r      = yolo_model(image)[0]
yolo_det    = sv.Detections.from_ultralytics(yolo_r)
solo_person = yolo_det[yolo_det.class_id == 0]
sam_bbox    = SAM(sam_path)
res_bbox    = sam_bbox(image, bboxes=solo_person.xyxy.tolist())[0]
det_bbox    = sv.Detections.from_ultralytics(res_bbox)

scn_texto = sv.MaskAnnotator(opacity=0.6).annotate(scene=image.copy(), detections=det_texto)
scn_bbox  = sv.MaskAnnotator(opacity=0.6).annotate(scene=image.copy(), detections=det_bbox)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
ax1.imshow(cv2.cvtColor(scn_texto, cv2.COLOR_BGR2RGB))
ax1.set_title(f'Texto: "person" ({len(det_texto)} objetos)')
ax1.axis("off")
ax2.imshow(cv2.cvtColor(scn_bbox, cv2.COLOR_BGR2RGB))
ax2.set_title(f"Bbox YOLO ({len(det_bbox)} objetos)")
ax2.axis("off")
plt.suptitle("Texto vs. bounding box", fontsize=13)
plt.tight_layout()
plt.show()
# 💭 Reflexión: ¿Cuál detecta más personas? ¿Las máscaras son igual de precisas?
# El texto puede encontrar personas que YOLO no detectó (mayor cobertura).
# Las cajas de YOLO tienden a producir máscaras más precisas en contornos.


## 🚀 Reto de extensión

**Tarea:** Para cada concepto (`"person"`, `"bus"`, `"wheel"`), encuentra
el objeto con mayor área de máscara y muéstralo en una figura de 1×3.

**Pista:**
```python
if detections.mask is not None:
    areas   = detections.mask.sum(axis=(1, 2))  # pixeles True por mascara
    idx_max = int(areas.argmax())
    mayor   = detections[idx_max]
```


In [ ]:
# Escribe tu solución aquí
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, concepto in zip(axes, ["person", "bus", "wheel"]):
    res = predictor(text=[concepto])[0]
    det = sv.Detections.from_ultralytics(res)
    if det.mask is not None and len(det) > 0:
        areas   = det.mask.sum(axis=(1, 2))   # área en píxeles de cada máscara
        idx_max = int(areas.argmax())          # índice del objeto más grande
        mayor   = det[idx_max]                 # sv.Detections con 1 objeto
        # annotated = mask_annotator.annotate(scene=image.copy(), detections=mayor)
        # ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.imshow(cv2.cvtColor(image.copy(), cv2.COLOR_BGR2RGB))
    ax.set_title(f'"{concepto}"')
    ax.axis("off")
plt.tight_layout()
plt.show()